?# Classification training

This notebook runs the training pipeline on synthetic recordings:

1. cut raw recordings into windows and preprocess each window independently;
2. encode every window with the frozen REVE-Base transformer (cached on disk);
3. tune an RBF SVM with Optuna and grouped cross-validation;
4. score the selected model once on held-out sessions;
5. write a run directory with metrics, figures and a Markdown report.

The synthetic generator is only a smoke test. Replace `recordings` with
`load_recordings("data/sessions")` to train on real data.

In [1]:
from pathlib import Path
import tempfile

from IPython.display import Markdown, display

from hotb_2026.classification.dataset import (
    build_dataset,
    make_synthetic_recording,
)
from hotb_2026.classification.training import TrainingConfig, train
from hotb_2026.features import ReveEncoder
from hotb_2026.reporting import write_run

/Users/dominikmika/PycharmProjects/HOTB_2026/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Build windows

Windows are sliced on the raw time axis and then preprocessed, so every
window is z-scored on its own — exactly what live inference will see.

A window takes the label that is active at its **start**. Windows whose label
changes before the **end** are dropped by default, which removes boundary
windows that belong to two states.

In [2]:
recordings = [
    make_synthetic_recording(
        duration_s=180.0,
        state_duration_s=30.0,
        seed=index,
        session_id=f"demo-{index + 1}",
    )
    for index in range(10)
]

# Ten sessions -> group_by="auto" keeps whole sessions intact.
dataset = build_dataset(recordings, window_s=10.0, group_by="auto")
dataset.summary()

{'n_windows': 180,
 'n_channels': 8,
 'n_samples': 2000,
 'sfreq': 200.0,
 'raw_sfreq': 250.0,
 'window_s': 10.0,
 'class_counts': {'low': 60, 'medium': 60, 'high': 60},
 'n_groups': 10,
 'groups': ['demo-1',
  'demo-10',
  'demo-2',
  'demo-3',
  'demo-4',
  'demo-5',
  'demo-6',
  'demo-7',
  'demo-8',
  'demo-9'],
 'label_names': ['low', 'medium', 'high'],
 'electrodes': ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']}

## 2. Train

REVE is frozen: embeddings are computed once and reused across Optuna
trials. The outer split holds out whole groups (sessions here) and is the
honest score. The inner grouped cross-validation drives the hyperparameter
search, so its score is optimistic.

In [3]:
# Use a persistent directory such as Path("runs/classification-demo") if you
# want to keep the artifacts after closing the notebook.
out_dir = Path(tempfile.mkdtemp(prefix="hotb-classification-"))

config = TrainingConfig(n_splits=5, n_trials=10, seed=0)
encoder = ReveEncoder(dataset.electrodes)

result = train(
    dataset,
    config=config,
    encoder=encoder,
    cache_dir=str(out_dir / "embeddings"),
)

print("CV     :", result["cv"])
print("Holdout:", result["holdout"]["metrics"])

W1114 23:30:42.362000 22322 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 1/1 [00:00<00:00, 4328.49it/s]


flash_attn not found, install it with `pip install flash_attn` if you want to use it


Loading weights: 100%|██████████| 140/140 [00:00<00:00, 9842.48it/s]


CV     : {'metric': 'balanced_accuracy', 'n_splits': 5, 'scores': [1.0, 0.9722222222222222, 1.0, 1.0, 1.0], 'mean': 0.9944444444444445, 'std': 0.011111111111111117}
Holdout: {'n_samples': 36, 'labels': [0, 1, 2], 'label_names': ['low', 'medium', 'high'], 'score': 1.0, 'metric': 'balanced_accuracy', 'accuracy': 1.0, 'balanced_accuracy': 1.0, 'f1_macro': 1.0, 'confusion_matrix': [[12, 0, 0], [0, 12, 0], [0, 0, 12]], 'per_class': {'low': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 12}, 'medium': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 12}, 'high': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 12}}, 'mean_confidence': 0.9728278204290005, 'mean_confidence_correct': 0.9728278204290005, 'mean_confidence_wrong': None, 'roc_auc_ovr': 1.0}


## 3. Write the report

`write_run` stores the configuration, dataset summary, Optuna trials, best
parameters, predictions (`predictions.npz`), figures and `report.md`.

Model serialization is deliberately **not** part of training — it lives in
the separate `model_store` step.

In [4]:
written = write_run(out_dir, result)
display(Markdown((out_dir / "report.md").read_text()))

# REVE + SVM training report

Generated: 2026-11-14T23:30:44

## Run

- Metric: `balanced_accuracy`
- Window: 10.000 s at 200.000 Hz
- Trials: 10 (best CV score 0.994)
- Groups: 8 train / 2 held out

## Dataset

- Windows: 180
- Channels: 8
- Samples per window: 2000
- Raw rate: 250.000 Hz
- Class counts: low: 60, medium: 60, high: 60

## Embeddings

- Encoder: `brain-bzh/reve-base`
- Shape: [180, 512]
- Cache: `/var/folders/8d/gk38hstj1gg1tk319gmlzgm00000gn/T/hotb-classification-1kyha660/embeddings`

## Hyperparameters

| parameter | value |
|---|---|
| `C` | 1.00000 |
| `gamma` | 0.00195 |
| `class_weight` | balanced |

Cross-validation was used to select these values, so its score is optimistic. The held-out score below is the honest estimate.

## Held-out test groups

- Windows: 36
- Groups: demo-1, demo-5
- Accuracy: 1.000
- Balanced accuracy: 1.000
- Macro F1: 1.000
- ROC AUC (OVR): 1.000
- Mean confidence: 0.973 (correct 0.973, wrong —)

### Confusion matrix

| true \ pred | low | medium | high |
|---|---|---|---|
| **low** | 12 | 0 | 0 |
| **medium** | 0 | 12 | 0 |
| **high** | 0 | 0 | 12 |

### Per class

| class | precision | recall | f1 | support |
|---|---|---|---|---|
| low | 1.000 | 1.000 | 1.000 | 12 |
| medium | 1.000 | 1.000 | 1.000 | 12 |
| high | 1.000 | 1.000 | 1.000 | 12 |

## Cross-validation (training groups)

- Folds: 5
- Scores: 1.000, 0.972, 1.000, 1.000, 1.000
- Mean: 0.994 ± 0.011

## Out-of-fold predictions

- Accuracy: 0.993
- Balanced accuracy: 0.993
- Macro F1: 0.993

## Files

- `config.json` — run configuration
- `dataset_summary.json` — windows, classes and groups
- `embeddings.meta.json` — encoder and embedding shape
- `trials.csv` — every Optuna trial
- `best_params.json` — selected hyperparameters
- `metrics.json` — CV, out-of-fold and held-out metrics
- `predictions.npz` — held-out and out-of-fold predictions
- `confusion_matrix.png`, `trials.png`, `embedding_pca.png` — figures


## Notes

- `result["model"]` is fitted on every window and is the artifact to save in
  the next step; `result["holdout_model"]` is the model used for the honest
  score.
- The final model is calibrated (`CalibratedClassifierCV` over `SVC`), so
  live inference can report a confidence for each class.
- Adjacent windows overlap in time, so splits must stay grouped. Never use a
  plain random split on window indices.